# Phase 4: explore the cleaned data

Run `python src/data_processing.py` first. It writes `data/processed/clean.csv`
and `data/processed/cleaning_log.json`, which this notebook reads.

**Reminder:** the target `xlogp` is PubChem's *computed* XLogP3, not a lab measurement.
Four questions here:

1. What did cleaning remove, and why?
2. What does the target look like?
3. What does it depend on? (And which feature must we keep out later?)
4. Do the extreme molecules look like real chemistry or like errors?

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, Draw

RDLogger.DisableLog("rdApp.*")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from data_processing import load_clean  # noqa: E402

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# ---- chart style: one blue, quiet chrome, no decoration -------------------
SURFACE = "#fcfcfb"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
BLUE = "#2a78d6"
# sequential ramp for density plots: blue steps 250 -> 700, light = few, dark = many
DENSITY = LinearSegmentedColormap.from_list(
    "blue_seq", ["#86b6ef", "#5598e7", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110,
})

df = load_clean()
log = json.loads((ROOT / "data" / "processed" / "cleaning_log.json").read_text())
print(f"clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. What did cleaning remove?

In [ ]:
steps = {s["step"]: s["rows_left"] for s in log["steps"]}
order = list(steps)
removed = {order[i]: steps[order[i - 1]] - steps[order[i]] for i in range(1, len(order))}

labels = {
    "has an XLogP value": "no XLogP value (no target)",
    "drop invalid_smiles": "invalid or empty SMILES",
    "drop multi_component": "several pieces (salts, solvates)",
    "drop bad_element": "unusual elements (metals etc.)",
    "drop too_heavy": f"weight above {log['max_mw']:g}",
    "merge duplicates + drop label conflicts": "duplicates of another molecule",
}
names = [labels[k] for k in removed]
counts = [removed[k] for k in removed]
idx = np.argsort(counts)  # largest bar ends up on top of a barh

kept = steps[order[-1]]
total = steps[order[0]]

fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.barh([names[i] for i in idx], [counts[i] for i in idx], height=0.55, color=BLUE,
        edgecolor=SURFACE, linewidth=1.5)
for y, i in enumerate(idx):
    ax.text(counts[i] + max(counts) * 0.012, y, str(counts[i]), va="center", color=INK, fontsize=11)
ax.set_xlim(0, max(counts) * 1.12)
ax.grid(axis="y", visible=False)
ax.set_xlabel("rows removed")
ax.set_title(f"Cleaning kept {kept:,} of {total:,} rows ({kept / total:.1%})", pad=12)
fig.tight_layout()
fig.savefig(RESULTS / "phase4_cleaning.png", dpi=160)
plt.show()

Most of what was removed simply had no XLogP: PubChem does not compute it for many
salts, metal compounds and mixtures. Everything else is small. Zero bars are shown on
purpose: they are rules that found nothing in *this* sample, but stay in for new data.

## 2. The target: XLogP3

In [ ]:
x = df["xlogp"]
lo, hi = -6, 12
inside = x[(x >= lo) & (x <= hi)]
outside = x[(x < lo) | (x > hi)]
median = x.median()

fig, ax = plt.subplots(figsize=(8.5, 4.2))
bins = np.arange(lo, hi + 0.5, 0.5)
counts_, _, _ = ax.hist(inside, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=1.2)
ax.set_ylim(0, counts_.max() * 1.22)  # headroom for the labels
ax.axvline(median, color=INK2, linestyle=(0, (4, 3)), linewidth=1.2)
ax.text(median + 0.2, counts_.max() * 1.17, f"median {median:.1f}", color=INK2, va="top", fontsize=11)
ax.grid(axis="x", visible=False)
ax.set_xlabel("XLogP3 (higher = prefers oil over water)")
ax.set_ylabel("molecules")
ax.set_title("Most molecules are moderately greasy; a few sit far out", pad=12)
ax.text(0.99, 0.97,
        f"{len(outside)} molecules outside {lo} to {hi}\n(min {x.min():.1f}, max {x.max():.1f})",
        transform=ax.transAxes, ha="right", va="top", color=INK2, fontsize=10)
fig.tight_layout()
fig.savefig(RESULTS / "phase4_target_hist.png", dpi=160)
plt.show()

print(x.describe().round(2).to_string())

## 3. What does it depend on?

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
feat = pd.DataFrame({
    "Molecular weight": df["mol_wt"].values,
    "Polar surface area (TPSA)": [Descriptors.TPSA(m) for m in mols],
    "RDKit Crippen logP": [Crippen.MolLogP(m) for m in mols],
})

view = (x >= lo) & (x <= hi)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), sharey=True)
hb = None
for ax, col in zip(axes, feat.columns):
    r = np.corrcoef(feat[col], x)[0, 1]
    hb = ax.hexbin(feat[col][view], x[view], gridsize=32, mincnt=1, cmap=DENSITY,
                   norm=LogNorm(vmin=1, vmax=60), linewidths=0.2, edgecolors=SURFACE)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel(col + (" (a second algorithm)" if "Crippen" in col else ""))
    ax.text(0.04, 0.95, f"r = {r:.2f}", transform=ax.transAxes, va="top", color=INK, fontsize=11,
            bbox=dict(facecolor=SURFACE, edgecolor="none", pad=3))
axes[0].set_ylabel("XLogP3")
cb = fig.colorbar(hb, ax=axes, pad=0.015, fraction=0.03)
cb.set_label("molecules per hexagon (log scale)", color=INK2)
cb.outline.set_edgecolor(AXIS)
fig.suptitle("Crippen logP tracks the target far more closely than size or polarity do",
             x=0.01, y=1.04, ha="left", fontsize=13, fontweight="bold")
fig.savefig(RESULTS / "phase4_dependence.png", dpi=160, bbox_inches="tight")
plt.show()

for col in feat.columns:
    print(f"{col:<28} correlation with XLogP3: {np.corrcoef(feat[col], x)[0, 1]:.3f}")

**Read this one carefully.** Crippen logP is a *different algorithm* for the same quantity
(both split the molecule into atom types and add up contributions). It correlates about
0.9 with our target, so it alone would explain roughly 80% of the variance. If we
fed it to a model as a feature, we would be handing over most of the answer, and the
score would say nothing about learning chemistry. **In Phase 6 we leave `MolLogP` out.**

## 4. Sanity check by eye: the extremes

In [ ]:
def grid(sub, title):
    print(title)
    legends = [f"CID {r.cid}   XLogP3 {r.xlogp:.1f}" for r in sub.itertuples()]
    return Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in sub["smiles"]], molsPerRow=3,
                                subImgSize=(300, 220), legends=legends)


def save_image(img, path):
    # Depending on the environment RDKit returns a PIL image or a notebook image.
    if hasattr(img, "save"):
        img.save(path)
    elif isinstance(img.data, bytes):
        Path(path).write_bytes(img.data)


top = grid(df.nlargest(6, "xlogp"), "Highest XLogP3 (most oil-loving)")
display(top)
save_image(top, RESULTS / "phase4_extremes_high.png")

bottom = grid(df.nsmallest(6, "xlogp"), "Lowest XLogP3 (most water-loving)")
display(bottom)
save_image(bottom, RESULTS / "phase4_extremes_low.png")

**What to look for.** The top group should be long hydrocarbon chains and big fatty molecules
(nothing polar to attract water). The bottom group should be full of acids, phosphates and
sugar-like groups (lots of oxygen and charge). If either group looked like ordinary
drug-sized molecules, we would suspect a bug. These extremes are *real chemistry*, so we
keep them, but they will dominate squared-error metrics such as RMSE. That is why we
report **MAE** next to RMSE in Phase 7.

## 5. Numbers to carry forward

In [ ]:
small = (df["n_heavy"] <= 8).sum()
print(f"rows                       : {len(df)}")
print(f"target range               : {x.min():.1f} to {x.max():.1f}  (middle 98%: {x.quantile(.01):.1f} to {x.quantile(.99):.1f})")
print(f"molecules with net charge  : {(df['net_charge'] != 0).sum()}")
print(f"tiny molecules (<= 8 heavy atoms): {small}  ({small / len(df):.1%}); ethanol has 3")
print(f"corr(XLogP3, Crippen logP) : {np.corrcoef(feat.iloc[:, 2], x)[0, 1]:.3f}")